# 宽动量实盘分析 — top5_rebal10_no_bond_wt_invvol

策略逻辑（对应 Grid Search 变体 `top5_rebal10_no_bond_wt_invvol`）：

1. **数据源** — `ETF_INDEX_MAP.get_all_symbols()` 全资产 ETF 池
2. **排除债券** — 剔除 cluster 43/44 债类标的
3. **因子筛选** — MA200 位置 ≥ 0, RSRS zscore ≥ 0, Trend R² ≥ 0.5
4. **排名** — 按近 20 日收益（跳过最近 1 日）降序
5. **选取 Top 5**
6. **持仓权重** — 反波动率加权（weight ∝ 1/σ_20d，归一化至 100%）

调仓频率为 10 个交易日。本 notebook 用于实盘决策：确认当期应持仓标的及各自权重比例。

In [1]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# 向上查找项目根（包含 libs/backtesting/__init__.py），确保任意 cwd 都能正确导入
def _find_project_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / "libs" / "backtesting" / "__init__.py").exists():
            return parent
    raise RuntimeError("找不到项目根目录 (libs/backtesting/__init__.py 不存在)")

PROJECT_ROOT = _find_project_root(Path.cwd())
LIBS_DIR = PROJECT_ROOT / "libs"
if str(LIBS_DIR) not in sys.path:
    sys.path.insert(0, str(LIBS_DIR))

from backtesting.wide_momentum_baseline import (
    ThresholdFilter,
    WideMomentumBaselineConfig,
    prepare_wide_momentum_universe,
)
from data_manager.providers.etf_index_map_provider import ETF_INDEX_MAP
from factors.price_return import PriceReturn
from factors.rsrs import RsrsFactor
from factors.trend_r2 import TrendR2Factor
from factors.ma import MAPosition
from factors.volatility import Volatility

print("导入完成")

导入完成


In [2]:
# ====================================================================
# 因子定义（与 run_wide_momentum_custom.py 策略参数一致）
# ====================================================================

pr_return = PriceReturn(window=20, skip_recent=1)          # 近 20 日收益（跳过最近 1 日）— 排名因子
trend_r2 = TrendR2Factor(window=120, output="r2")         # 120 日趋势 R²
rsrs = RsrsFactor(
    regression_window=14, zscore_window=600, output="zscore"
)                                                         # RSRS zscore
ma200 = MAPosition(window=200, price_column="close")       # close 相对 MA200 的位置
vol20 = Volatility(window=20)                              # 20 日波动率（反波动率加权用）

# 因子输出列名
RANKING_COL = pr_return.get_output_name()
RSRS_COL = rsrs.get_output_name()
TREND_R2_COL = trend_r2.get_output_name()
MA_POS_COL = ma200.get_output_name()
VOL_COL = vol20.get_output_name()

# 策略参数
TOP_N = 5
BOND_CLUSTERS = (43, 44)

print(f"排序因子列:  {RANKING_COL}")
print(f"波动率列:    {VOL_COL}")
print(f"筛选条件:")
print(f"  {MA_POS_COL} >= 0")
print(f"  {RSRS_COL} >= 0")
print(f"  {TREND_R2_COL} >= 0.5")
print(f"选取: Top {TOP_N}, 排除债券 (cluster {BOND_CLUSTERS})")
print(f"权重: 反波动率加权 (1/{VOL_COL})")

排序因子列:  PriceReturn_20_skip1
波动率列:    Volatility_20
筛选条件:
  MAPosition_close_200 >= 0
  RSRS_zscore_adj >= 0
  TrendR2_120_r2 >= 0.5
选取: Top 5, 排除债券 (cluster (43, 44))
权重: 反波动率加权 (1/Volatility_20)


In [3]:
# ====================================================================
# 准备 universe：加载数据 + 计算因子
# 数据源: ETF_INDEX_MAP.get_all_symbols()
# ====================================================================

config = WideMomentumBaselineConfig(
    ranking_factor=pr_return,
    factor_pipeline=(rsrs, pr_return, trend_r2, ma200, vol20),
    builtin_filters=(),      # notebook 中手工筛选，不依赖内置过滤器
    end_date=None,            # 自动取最新数据截止日
    min_listing_days=0,
)

symbols = ETF_INDEX_MAP.get_all_symbols()
print(f"数据源: ETF_INDEX_MAP ({len(symbols)} 标的)")
print("加载数据并计算因子（可能需要几分钟）...")

prepared = prepare_wide_momentum_universe(config=config, symbols=symbols)

print(f"\nUniverse 就绪:")
print(f"  原始来源标的:  {prepared.source_symbol_count}")
print(f"  最终保留标的:  {len(prepared.symbol_data_map)}")
print(f"  数据范围:      {prepared.start_date.date()} → {prepared.end_date.date()}")
print(f"  最新完整日:    {prepared.recent_complete_date.date()}")

# 排除原因统计
from collections import Counter
reason_counts = Counter(prepared.excluded_symbols.values()) if prepared.excluded_symbols else {}
load_error_count = len(prepared.load_errors) if prepared.load_errors else 0

print(f"\n被排除标的诊断:")
print(f"  数据加载失败:  {load_error_count}")
for reason, count in reason_counts.most_common():
    print(f"  {reason}: {count}")
dropped = prepared.source_symbol_count - len(prepared.symbol_data_map)
print(f"  合计排除:      {dropped} / {prepared.source_symbol_count}")

数据源: ETF_INDEX_MAP (483 标的)
加载数据并计算因子（可能需要几分钟）...

Universe 就绪:
  原始来源标的:  481
  最终保留标的:  478
  数据范围:      2023-12-04 → 2026-06-23
  最新完整日:    2026-06-23

被排除标的诊断:
  数据加载失败:  0
  stale_before_recent_complete_date: 3
  合计排除:      3 / 481


In [4]:
# ====================================================================
# 筛选逻辑：排除债券 → 三因子过滤 → 按收益排序 → 取 Top 5
# ====================================================================

latest_date = prepared.recent_complete_date

rows: list[dict] = []
excluded_bonds: list[str] = []
excluded_eligibility: list[str] = []
excluded_rsrs: list[str] = []
excluded_trend_r2: list[str] = []
excluded_ma_pos: list[str] = []

for symbol, sd in prepared.symbol_data_map.items():
    # 1. 排除债券
    if sd.cluster_label in BOND_CLUSTERS:
        excluded_bonds.append(symbol)
        continue

    # 2. 取最新日期的因子行
    frame = sd.frame
    if latest_date not in frame.index:
        continue
    row = frame.loc[latest_date]

    # 3. eligible_signal 检查
    if not bool(row.get("eligible_signal", False)):
        excluded_eligibility.append(symbol)
        continue

    # 4. 因子值提取
    rsrs_val = float(row.get(RSRS_COL, np.nan))
    trend_r2_val = float(row.get(TREND_R2_COL, np.nan))
    ma_pos_val = float(row.get(MA_POS_COL, np.nan))
    vol_val = float(row.get(VOL_COL, np.nan))
    ranking_val = float(row.get("ranking_value", np.nan))
    latest_close = float(row.get("close", np.nan))

    # 5. 三因子筛选
    if pd.isna(ma_pos_val) or ma_pos_val < 0:
        excluded_ma_pos.append(symbol)
        continue
    if pd.isna(rsrs_val) or rsrs_val < 0:
        excluded_rsrs.append(symbol)
        continue
    if pd.isna(trend_r2_val) or trend_r2_val < 0.5:
        excluded_trend_r2.append(symbol)
        continue

    # 通过所有筛选
    etf_name = ""
    if sd.etf_data is not None:
        etf_name = getattr(sd.etf_data, "name", "") or ""

    rows.append({
        "symbol": symbol,
        "name": etf_name,
        "cluster_label": sd.cluster_label,
        "ranking_value": round(ranking_val, 6),
        RSRS_COL: round(rsrs_val, 4),
        TREND_R2_COL: round(trend_r2_val, 4),
        MA_POS_COL: round(ma_pos_val, 4),
        VOL_COL: round(vol_val, 6) if not pd.isna(vol_val) else np.nan,
        "latest_close": round(latest_close, 4),
    })

# 转 DataFrame 并排序
result_df = pd.DataFrame(rows)
if not result_df.empty:
    result_df = result_df.sort_values("ranking_value", ascending=False).reset_index(drop=True)
    result_df.index = result_df.index + 1  # 排名从 1 开始
    result_df.index.name = "rank"

# 筛选统计
total = len(prepared.symbol_data_map)
passed = len(result_df)
print(f"筛选统计（总计 {total} 标的，{prepared.source_symbol_count} 原始来源）:")
print(f"  排除债券 (cluster {BOND_CLUSTERS}): {len(excluded_bonds)}")
print(f"  不满足 eligible: {len(excluded_eligibility)}")
print(f"  MAPosition < 0: {len(excluded_ma_pos)}")
print(f"  RSRS < 0:       {len(excluded_rsrs)}")
print(f"  TrendR2 < 0.5:  {len(excluded_trend_r2)}")
print(f"  通过筛选:       {passed}")
print(f"\n数据截止日期: {latest_date.date()}")

筛选统计（总计 478 标的，481 原始来源）:
  排除债券 (cluster (43, 44)): 12
  不满足 eligible: 0
  MAPosition < 0: 253
  RSRS < 0:       67
  TrendR2 < 0.5:  81
  通过筛选:       65

数据截止日期: 2026-06-23


In [5]:
# ====================================================================
# 选取 Top 5 + 反波动率加权
# 权重 = (1/vol_i) / Σ(1/vol_j)，j ∈ top5
# ====================================================================

top_df = result_df.head(TOP_N).copy()

# 计算反波动率权重
FLOOR = 1e-8
inv_vol = top_df[VOL_COL].apply(lambda v: 1.0 / max(abs(v), FLOOR) if not pd.isna(v) else FLOOR)
inv_vol_sum = inv_vol.sum()
top_df["weight"] = (inv_vol / inv_vol_sum).round(4)
top_df["weight_pct"] = (top_df["weight"] * 100).round(2)

print(f"═" * 60)
print(f"  策略: top5_rebal10_no_bond_wt_invvol")
print(f"  数据截止: {latest_date.date()}")
print(f"  通过筛选标的数: {passed}，选取 Top {TOP_N}")
print(f"═" * 60)
print()

display_cols = ["symbol", "name", "cluster_label", "ranking_value",
                VOL_COL, "weight_pct",
                RSRS_COL, TREND_R2_COL, MA_POS_COL, "latest_close"]
display_cols = [c for c in display_cols if c in top_df.columns]

print(f"当期持仓（Top {TOP_N}，反波动率加权）:")
print()
top_df[display_cols]

════════════════════════════════════════════════════════════
  策略: top5_rebal10_no_bond_wt_invvol
  数据截止: 2026-06-23
  通过筛选标的数: 65，选取 Top 5
════════════════════════════════════════════════════════════

当期持仓（Top 5，反波动率加权）:



,symbol,name,cluster_label,ranking_value,Volatility_20,weight_pct,RSRS_zscore_adj,TrendR2_120_r2,MAPosition_close_200,latest_close
rank,,,,,,,,,,
1,515880,通信ETF国泰,8,0.235216,0.036940,19.30,0.0615,0.8100,0.3863,5.382
2,588010,科创新材料ETF博时,14,0.227312,0.036588,19.48,0.5843,0.6105,0.3178,1.361
3,159516,半导体设备ETF国泰,6,0.226209,0.040694,17.52,0.7830,0.5824,0.4419,3.136
4,588020,科创成长ETF易方达,5,0.225842,0.033167,21.49,1.0863,0.6840,0.3526,3.269
5,513520,日经ETF华夏,41,0.222326,0.032108,22.20,0.1240,0.6354,0.2144,2.447


In [9]:
# ====================================================================
# 持仓汇总：简洁视图（适合实盘参考）
# ====================================================================

print(f"\n{'─' * 50}")
print(f"  实盘持仓建议 ({latest_date.date()})")
print(f"  调仓周期: 每 10 个交易日")
print(f"{'─' * 50}")
print(f"{'排名':<4} {'代码':<8} {'名称':<16} {'权重%':>6} {'最新价':>8}")
print(f"{'─' * 50}")

for idx, row in top_df.iterrows():
    print(f"{idx:<4} {row['symbol']:<8} {row['name']:<16} {row['weight_pct']:>6.2f} {row['latest_close']:>8.4f}")

print(f"{'─' * 50}")
print(f"  权重合计: {top_df['weight_pct'].sum():.2f}%")
print()

# 如有总投资金额，可计算各标的应买金额
TOTAL_CAPITAL = 550000  # 修改为实际投资金额（元）
print(f"  参考投入金额（总资金 {TOTAL_CAPITAL:,.0f} 元）:")
for idx, row in top_df.iterrows():
    amount = TOTAL_CAPITAL * row['weight']
    shares = int(amount / row['latest_close'] / 100) * 100  # ETF 以 100 份为单位
    print(f"    {row['symbol']} {row['name']}: {amount:>10,.2f} 元 ≈ {shares} 份")


──────────────────────────────────────────────────
  实盘持仓建议 (2026-06-23)
  调仓周期: 每 10 个交易日
──────────────────────────────────────────────────
排名   代码       名称                  权重%      最新价
──────────────────────────────────────────────────
1    515880   通信ETF国泰           19.30   5.3820
2    588010   科创新材料ETF博时        19.48   1.3610
3    159516   半导体设备ETF国泰        17.52   3.1360
4    588020   科创成长ETF易方达        21.49   3.2690
5    513520   日经ETF华夏           22.20   2.4470
──────────────────────────────────────────────────
  权重合计: 99.99%

  参考投入金额（总资金 550,000 元）:
    515880 通信ETF国泰: 106,150.00 元 ≈ 19700 份
    588010 科创新材料ETF博时: 107,140.00 元 ≈ 78700 份
    159516 半导体设备ETF国泰:  96,360.00 元 ≈ 30700 份
    588020 科创成长ETF易方达: 118,195.00 元 ≈ 36100 份
    513520 日经ETF华夏: 122,100.00 元 ≈ 49800 份


In [10]:
# ====================================================================
# 全部通过筛选的标的（参考）
# ====================================================================

display_all_cols = ["symbol", "name", "cluster_label", "ranking_value",
                    RSRS_COL, TREND_R2_COL, MA_POS_COL, VOL_COL, "latest_close"]
display_all_cols = [c for c in display_all_cols if c in result_df.columns]

print(f"全部通过筛选标的 (前 30 / 共 {len(result_df)}):")
result_df[display_all_cols].head(30)

全部通过筛选标的 (前 30 / 共 65):


,symbol,name,cluster_label,ranking_value,RSRS_zscore_adj,TrendR2_120_r2,MAPosition_close_200,Volatility_20,latest_close
rank,,,,,,,,,
1,515880,通信ETF国泰,8,0.235216,0.0615,0.8100,0.3863,0.036940,5.382
2,588010,科创新材料ETF博时,14,0.227312,0.5843,0.6105,0.3178,0.036588,1.361
3,159516,半导体设备ETF国泰,6,0.226209,0.7830,0.5824,0.4419,0.040694,3.136
4,588020,科创成长ETF易方达,5,0.225842,1.0863,0.6840,0.3526,0.033167,3.269
5,513520,日经ETF华夏,41,0.222326,0.1240,0.6354,0.2144,0.032108,2.447
6,513880,日经225ETF华安,41,0.216121,0.3226,0.6183,0.2127,0.031902,2.274
7,159967,创业板成长ETF华夏,20,0.196584,0.5569,0.7604,0.3364,0.033652,3.697
8,159811,5GETF博时,8,0.192566,0.6642,0.7641,0.3498,0.034008,3.750
9,515000,科技ETF华宝,7,0.187221,0.8567,0.6522,0.3127,0.030776,3.104


In [11]:
# ====================================================================
# 导出为 xlsx
# ====================================================================
import os
from config import DataPath

output_dir = Path(os.path.join(DataPath.DEFAULT_WINDOWS_PATH, "top5_rebal10_no_bond_wt_invvol"))
output_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"portfolio_{ts}.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Sheet 1: 当期持仓（Top 5 + 权重）
    portfolio_export = top_df.copy()
    portfolio_export.insert(0, "rank", portfolio_export.index)
    portfolio_export.to_excel(writer, sheet_name="当期持仓", index=False)

    # Sheet 2: 全部通过筛选标的
    all_export = result_df.copy()
    all_export.insert(0, "rank", all_export.index)
    all_export.to_excel(writer, sheet_name="全部筛选结果", index=False)

    # Sheet 3: 统计信息
    summary_data = {
        "指标": ["策略标签", "数据截止日期", "原始标的数", "有效标的数",
                 "排除债券", "排除(eligibility)", "排除(MAPosition<0)",
                 "排除(RSRS<0)", "排除(TrendR2<0.5)",
                 "通过筛选", "选取 Top N", "权重方案"],
        "值": ["top5_rebal10_no_bond_wt_invvol",
               str(latest_date.date()), prepared.source_symbol_count, total,
               len(excluded_bonds), len(excluded_eligibility),
               len(excluded_ma_pos), len(excluded_rsrs), len(excluded_trend_r2),
               passed, TOP_N, "反波动率加权 (1/Volatility_20)"],
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="统计信息", index=False)

print(f"导出完成: {output_path}")
print(f"当期持仓 {TOP_N} 标的，全部筛选 {len(result_df)} 标的")

导出完成: /mnt/c/Users/wyg/Documents/invest/top5_rebal10_no_bond_wt_invvol/portfolio_20260623_213022.xlsx
当期持仓 5 标的，全部筛选 65 标的
